# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udayk-25/FLYRANK/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I have selected **Lane 2: Refresh / Content Opportunity Scoring** as my provisional lane.

**ML Task Type: Ranking and Scoring**

**Why?**
The client's content editorial team has a finite capacity (e.g., they can only review and update 50 pages a week). A simple binary classification model that outputs a "yes/no" label for decline does not help the team prioritize their workload because it results in a huge, unsorted block of flagged pages. By outputting a continuous risk/opportunity score (from 0 to 100), we can rank the pages and select the top K pages that have the highest probability of decay and highest traffic exposure. This allows the team to act on the most urgent pages first.


In [1]:
# No code required here; task type framing is described in the markdown cell above.


## 2. Target or proxy

**What we predict:**
We predict **observed future traffic decline**.

**Where the label comes from:**
- In the starter dataset, we use a proxy target: `is_declining_label = trend_direction == 'down'` (which is derived from `trend_pct` representing the traffic change in the current window).
- In the full warehouse release, we define the target as an **observed outcome in a future window**: whether a page's organic search traffic (clicks/impressions) drops by more than 20% in the 30-day window *following* the 90-day feature collection window.
- Crucially, this target is **observed from real GSC performance logs**, not defined by a manual rule, ensuring the model learns actual search market changes rather than hardcoded heuristics.


In [2]:
# No code required here; target/proxy framing is described in the markdown cell above.


## 3. Success metric

**Our chosen metric:**
**Precision@50** (of the top 50 recommended pages, what fraction are actually declining).

**Defense of the metric:**
Precision@50 directly matches the operational capacity of the client's editorial team. If the team can only review 50 pages in a batch, the priority score is only useful if the top-ranked pages are indeed declining. If Precision@50 is low (e.g., 20%), editors waste 80% of their time reviewing stable pages. If Precision@50 is high (e.g., 75%), editors spend their time productively on pages in real need of refresh. We prioritize Precision@K over generic accuracy because accuracy evaluates the entire inventory, whereas editors only care about the top of the recommended queue.


In [3]:
# No code required here; success metric framing is described in the markdown cell above.


## 4. The unit of analysis, as a real dataframe

The **unit of analysis** is **one unique webpage (pseudonymized content item) associated with a client**.

Below, we load the starter dataset and display the unit of analysis, showing the unique keys (`content_id`, `client_id`), key observed features (`impressions_90d`, `days_since_last_update`, `ctr`, `avg_position`), and our target indicator (`trend_direction`).


In [4]:
import pandas as pd

# Load starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Show the unit of analysis: one row = one content item per client
print(f"DataFrame Shape: {df.shape[0]} rows (units) x {df.shape[1]} columns")
df[['content_id', 'client_id', 'impressions_90d', 'days_since_last_update', 'ctr', 'avg_position', 'trend_direction']].head(5)


DataFrame Shape: 30000 rows (units) x 44 columns


,content_id,client_id,impressions_90d,days_since_last_update,ctr,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,0.76,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,0.05,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,0.09,36.5,down
3,content_331d6c4de07b,client_19581e27de,11751,22,0.49,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,0.13,44.0,down


## 5. Why ML beats a fixed rule here

**Why a fixed rule fails:**
A simple if-statement (like `days_since_last_update >= 180`) is too rigid and fails to capture the complex, multi-dimensional nature of traffic decay. 
- **Non-linear interactions**: A page that is 200 days old but holds a stable position 1 rank and captures a very high, stable CTR does not need a refresh. Conversely, a page that is only 90 days old but is rapidly sliding in average position (e.g. from 3.2 to 8.7) and losing CTR needs urgent attention.
- **The CTR Cliff**: Click-through rates collapse non-linearly as rankings drop. A drop in position from 1 to 3 has a massive CTR impact compared to a drop from 11 to 13 due to page layout boundaries. Simple rules cannot easily map these non-linear thresholds.
- **Feature complexity**: Position decay, CTR drops, search volume changes, and content length interact in complex ways. A machine learning model (such as a Random Forest or Gradient Boosted Trees) can learn these non-linear decision boundaries and feature interaction terms, significantly beating any manually maintained nesting of if-else statements.


In [5]:
# No code required here; ML vs rule explanation is described in the markdown cell above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
